# Local, global and naive search

The three engines answer the same question from different evidence:

- **`LocalSearchEngine`** retrieves the entities closest to the query and walks out
  to their relations, source chunks and community summaries. Best for questions
  about specific things.
- **`GlobalSearchEngine`** reads community summaries instead of entities, so it
  answers questions about the corpus as a whole. It needs a graph built with
  `make_community_summary=True`.
- **`NaiveSearchEngine`** skips the graph entirely and does plain vector search over
  chunks. It is the baseline the graph engines are measured against.

**Environment:** `OPENAI_API_KEY`, `LLM_MODEL_NAME`, `EMBEDDER_MODEL_NAME`, and
optionally `OPENAI_BASE_URL` (defaults to `https://api.openai.com/v1`).

In [ ]:
import os
from pathlib import Path

from ragu import (
    ArtifactsExtractorLLM,
    BuilderArguments,
    GlobalSearchEngine,
    KnowledgeGraph,
    LocalSearchEngine,
    NaiveSearchEngine,
    Settings,
    SimpleChunker,
)
from ragu.models.embedder import EmbedderOpenAI
from ragu.models.llm import LLMOpenAI
from ragu.models.openai import CachedAsyncOpenAI
from ragu.search_engine.local_search import LocalParams
from ragu.search_engine.naive_search import NaiveSearchParams
from ragu.utils.ragu_utils import read_text_from_files

DATA_DIR = Path("data/en")
QUESTION = "Where did the father of the creator of the C programming language work?"

## Models

One `CachedAsyncOpenAI` is shared by the LLM and the embedder, so rate limiting,
retries and caching are enforced once for the whole notebook.

In [ ]:
Settings.language = "english"
Settings.storage_folder = "ragu_working_dir/search_engines_example"

client = CachedAsyncOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    api_key=os.environ["OPENAI_API_KEY"],
    rate_max_simultaneous=10,
    rate_max_per_minute=100,
)
llm = LLMOpenAI(client=client, model_name=os.environ["LLM_MODEL_NAME"])
embedder = EmbedderOpenAI(client=client, model_name=os.environ["EMBEDDER_MODEL_NAME"])
await embedder.initialize()

## Build the graph

This is the expensive cell — it runs extraction and summarization over the corpus.
Run it once, then iterate on the query cells below as often as you like.

In [ ]:
knowledge_graph = KnowledgeGraph(
    llm=llm,
    embedder=embedder,
    chunker=SimpleChunker(max_chunk_size=1000),
    artifact_extractor=ArtifactsExtractorLLM(llm=llm, embedder=embedder),
    builder_settings=BuilderArguments(make_community_summary=True),
)
await knowledge_graph.build_from_docs(read_text_from_files(DATA_DIR))

## Local search

`use_summary` pulls in community summaries of the retrieved entities; `use_chunks`
(on by default) pulls in the raw text they were extracted from.

In [ ]:
local = LocalSearchEngine(llm=llm, knowledge_graph=knowledge_graph, embedder=embedder)
response = await local.query(QUESTION, LocalParams(top_k=10, use_summary=True))

print(response.response)
print(f"\nentities: {len(response.retrieval.result.entities)}, "
      f"relations: {len(response.retrieval.result.relations)}")

## Global search

Global search takes no retrieval parameters: its context is the whole set of
community summaries, rated and filtered by the LLM itself.

In [ ]:
global_search = GlobalSearchEngine(llm=llm, knowledge_graph=knowledge_graph)
response = await global_search.query(QUESTION)

print(response.response)
print(f"\ncommunity insights: {len(response.retrieval.result.insights)}")

## Naive search

No graph involved — just vector similarity over chunks.

In [ ]:
naive = NaiveSearchEngine(llm=llm, knowledge_graph=knowledge_graph, embedder=embedder)
response = await naive.query(QUESTION, NaiveSearchParams(top_k=10))

print(response.response)
print(f"\nchunks: {len(response.retrieval.result.chunks)}")

## Inspect the retrieval

Every response carries the context it was generated from, so you can check *why*
an answer came out the way it did without re-running the query.

In [ ]:
retrieval = (await local.search(QUESTION, LocalParams(top_k=5))).result
for entity in retrieval.entities:
    print(f"{entity.entity_name} ({entity.entity_type}): {entity.description[:100]}")

## Customizing the prompt

Every engine owns the prompts it uses and exposes them through three methods
inherited from `RaguGenerativeModule`: `get_prompts()`, `get_prompt(name)` and
`update_prompt(name, instruction)`.

Ask an engine what it owns rather than guessing — `LocalSearchEngine` registers one
prompt, `GlobalSearchEngine` two.

In [ ]:
print(f"local:  {list(local.get_prompts())}")
print(f"global: {list(global_search.get_prompts())}")
print(f"naive:  {list(naive.get_prompts())}")

instruction = local.get_prompt("local_search")
print(f"\nschema: {instruction.pydantic_model.__name__}")
print(f"description: {instruction.description}\n")
for message in instruction.messages:
    print(f"--- [{message.role}] ---")
    print(message.content.strip())

`RAGUInstruction` is a frozen dataclass, so replace it rather than mutate it.
`dataclasses.replace` keeps the output schema and the few-shot formatter while
swapping the templates.

A template may only use the variables the engine passes — `query`, `context` and
`language` for all three engines here. The Jinja environment uses `StrictUndefined`,
so a typo raises instead of rendering an empty string.

Updates are scoped to the instance: the global `DEFAULT_PROMPT_TEMPLATES` registry
and every other engine are untouched, which is what makes an A/B comparison
possible.

In [ ]:
import dataclasses

from ragu.common.prompts import ChatMessages, SystemMessage, UserMessage

tuned = LocalSearchEngine(llm=llm, knowledge_graph=knowledge_graph, embedder=embedder)
tuned.update_prompt(
    "local_search",
    dataclasses.replace(
        local.get_prompt("local_search"),
        messages=ChatMessages.from_messages([
            SystemMessage("You are a terse analyst. Prefer relations over prose."),
            UserMessage(
                "Question: {{ query }}\n\nContext:\n{{ context }}\n\n"
                "Answer in {{ language }} in at most two sentences, then list the "
                "entities you relied on in brackets."
            ),
        ]),
    ),
)

print("=== default ===")
print((await local.query(QUESTION, LocalParams(top_k=10))).response)
print("\n=== custom ===")
print((await tuned.query(QUESTION, LocalParams(top_k=10))).response)

Setting `pydantic_model` on the instruction turns the engine into a structured
extractor — `response.response` becomes a validated model instead of a string.
See **`custom_prompts_example.ipynb`** for that, for the full prompt registry, and
for the gotchas (streaming ignores the schema; other modules validate theirs).

## Clean up

Releases connection pools held by the storage backends.

In [ ]:
await knowledge_graph.index.close()